# 5. Advanced layers — Loop, RNN, and If

`Loop_controller`, `Rnn`, and `If_controller` are containers. At their core you
can build and connect them like any other node in the graph.

It is **not** recommended to use these controllers as the top-level model. 
If required, wrap them with a thin DAG for consistency.


In [1]:
from pathlib import Path

import numpy as np
from fpga_ai_suite.spatial.flow import run_flow
from fpga_ai_suite.spatial.layers import (
    Activation,
    Add,
    Add_bias,
    Constant,
    Directed_acyclic_graph,
    Dot,
    If_controller,
    Loop_controller,
    Rnn,
    Saturation,
    Sequential,
)

WORK = Path("work_advanced_layers")
WORK.mkdir(exist_ok=True)


## `Loop_controller`

Fixed iteration count. The body must expose ports named `din` and `dout`, and
keep the same shape each iteration. `Sequential` is a convenient body builder
for single-I/O layer chains.


In [2]:
class Loop_outer(Directed_acyclic_graph):
    def __init__(self, weights, bias, num_iters=3, num_lanes=4):
        self.weights_data = weights
        self.bias_data = bias
        self.num_iters = num_iters
        self.num_lanes = num_lanes
        super().__init__(module_name="Loop_outer")

    def init_topology(self):
        self.input = self.create_input("din", num_lanes=self.num_lanes, dtype="BF16")
        self.output = self.create_output("dout", num_lanes=self.num_lanes, dtype="BF16")

        body = Sequential(
            name="loop_body",
            layers=[
                Dot(
                    weights=self.weights_data,
                    num_lanes_in=self.num_lanes,
                    num_lanes_out=self.num_lanes,
                    dtype="BF16",
                    mif_init=True,
                ),
                Add_bias(
                    bias=self.bias_data,
                    num_lanes=self.num_lanes,
                    dtype="BF16",
                    mif_init=True,
                ),
                Activation(activation="relu", num_lanes=self.num_lanes, dtype="BF16"),
            ],
        )
        self.loop = self.add_node(
            Loop_controller(
                loop_body=body,
                num_iters=self.num_iters,
                num_lanes=self.num_lanes,
                dw=16,
                dtype="BF16",
            )
        )
        self.connect(self.input, self.loop.inputs.din)
        self.connect(self.loop.outputs.dout, self.output)


loop_w = np.eye(128, dtype=np.float32)
loop_b = np.ones(128, dtype=np.float32)
loop_model = Loop_outer(loop_w, loop_b)
loop_in = np.ones((2, 1, 128), dtype=np.float32)
loop_out, = loop_model(din=loop_in)
print("loop out shape:", loop_out.shape)


loop out shape: (2, 1, 128)


## `Rnn`

Scans a sequence. Pass the body DAG plus lists of **body-owned** ports for
recurrent state (`states_in` / `states_out`). The controller exposes `din`,
`initial_states` (a bus), and `dout`.

Below: a tiny cell that adds the input token to a running state.


In [3]:
class Tiny_rnn_cell(Directed_acyclic_graph):
    def __init__(self, num_lanes=4, dw=16):
        self.num_lanes = num_lanes
        self.dw = dw
        super().__init__(module_name="Tiny_rnn_cell")

    def init_topology(self):
        self.din = self.create_input("din", self.dw, self.num_lanes, "BF16")
        self.hin = self.create_input("hin", self.dw, self.num_lanes, "BF16")
        self.dout = self.create_output("dout", self.dw, self.num_lanes, "BF16")
        self.hout = self.create_output("hout", self.dw, self.num_lanes, "BF16")

        self.add = self.add_node(Add(num_lanes=self.num_lanes, dw=self.dw, dtype="BF16"))
        self.connect(self.din, self.add.inputs.dina)
        self.connect(self.hin, self.add.inputs.dinb)
        self.connect(self.add.outputs.dout, self.dout)
        self.connect(self.add.outputs.dout, self.hout)


class Rnn_outer(Directed_acyclic_graph):
    def __init__(self, max_len=4, units=4, num_lanes=4):
        self.max_len = max_len
        self.units = units
        self.num_lanes = num_lanes
        super().__init__(module_name="Rnn_outer")

    def init_topology(self):
        self.input = self.create_input("din", 16, self.num_lanes, "BF16")
        self.output = self.create_output("dout", 16, self.num_lanes, "BF16")

        cell = Tiny_rnn_cell(num_lanes=self.num_lanes)
        self.rnn = self.add_node(
            Rnn(
                rnn_body=cell,
                max_len=self.max_len,
                output_shape=(1, self.units),
                states_in=[cell.hin],
                states_out=[cell.hout],
                num_lanes_in=self.num_lanes,
                num_lanes=self.num_lanes,
                dw=16,
                dtype="BF16",
            )
        )
        # Zero initial state, paced by the sequence input
        self.h0 = self.add_node(
            Constant(value=0, m=self.units, n=1, num_lanes=self.num_lanes, dw=16, dtype="BF16")
        )
        self.register_rom_dependency(self.h0, self.input)
        self.connect(self.h0.outputs.dout, self.rnn.inputs.initial_states[0])
        self.connect(self.input, self.rnn.inputs.din)
        self.connect(self.rnn.outputs.dout, self.output)


rnn_model = Rnn_outer()
# din packets must be a multiple of max_len; each packet is (1, units)
rnn_in = np.ones((4, 1, 4), dtype=np.float32)
rnn_out, = rnn_model(din=rnn_in)
print("rnn out:", rnn_out)


rnn out: [[[1. 1. 1. 1.]]

 [[2. 2. 2. 2.]]

 [[3. 3. 3. 3.]]

 [[4. 4. 4. 4.]]]


## `If_controller`

Per-packet branch. Wire:

- `control` — `UINT` with shape `(P, 1, 1)` (lanes=1); `1` selects the then body
- `din` / `dout` — data path

Both bodies need `din` and `dout`. Unlike Loop, in/out widths and dtypes may
differ between branches (configure via constructor args).


In [4]:
class Act_body(Directed_acyclic_graph):
    def __init__(self, activation, name):
        self.activation = activation
        self._name = name
        super().__init__(module_name=name)

    def init_topology(self):
        self.din = self.create_input("din", 16, 4, "BF16")
        self.dout = self.create_output("dout", 16, 4, "BF16")
        self.act = self.add_node(
            Activation(activation=self.activation, num_lanes=4, dw=16, dtype="BF16")
        )
        self.connect(self.din, self.act.inputs.din)
        self.connect(self.act.outputs.dout, self.dout)


class If_outer(Directed_acyclic_graph):
    def __init__(self):
        super().__init__(module_name="If_outer")

    def init_topology(self):
        self.input = self.create_input("din", 16, 4, "BF16")
        self.control = self.create_input("control", dw=1, num_lanes=1, dtype="UINT")
        self.output = self.create_output("dout", 16, 4, "BF16")

        self.branch = self.add_node(
            If_controller(
                then_body=Act_body("relu", "then_relu"),
                else_body=Act_body("sigmoid", "else_sigmoid"),
                num_lanes_in=4,
                num_lanes_out=4,
                dw_in=16,
                dw_out=16,
                dtype_in="BF16",
                dtype_out="BF16",
            )
        )
        self.connect(self.input, self.branch.inputs.din)
        self.connect(self.control, self.branch.inputs.control)
        self.connect(self.branch.outputs.dout, self.output)


if_model = If_outer()
P = 4
din = np.linspace(-1, 1, P * 4, dtype=np.float32).reshape(P, 1, 4)
# Alternate then / else
ctrl = np.array([1, 0, 1, 0], dtype=np.uint32).reshape(P, 1, 1)
if_out, = if_model(din=din, control=ctrl)
print("if out:", if_out)


if out: [[[0.         0.         0.         0.        ]]

 [[0.38476562 0.41796875 0.44921875 0.48242188]]

 [[0.06689453 0.20019531 0.33398438 0.46679688]]

 [[0.64453125 0.67578125 0.703125   0.73046875]]]


## Optional: simulate the Loop example


In [5]:
loop_model(din=loop_in)
ret = run_flow(loop_model, sim=True)


Dumping input and output data...
Dumping weights: matrix size: [128,128] total weights: 8192
Dumping bias weights. Total weights: 128
Running model LOOP_OUTER

Running Simulation (vcs)...
Running VCS in sim_LOOP_OUTER (device_family=Agilex 7)
*** Using c compiler gcc instead of cc ...
                         Chronologic VCS (TM)
      Version V-2023.12-SP2-1_Full64 -- Fri Sep  4 12:31:32 2026

                    Copyright (c) 1991 - 2024 Synopsys, Inc.
   This software and the associated documentation are proprietary to Synopsys,
 Inc. This software may only be used in accordance with the terms and conditions
 of a written license agreement with Synopsys, Inc. All other use, reproduction,
   or distribution of this software is strictly prohibited.  Licensed Products
     communicate with Synopsys servers for the purpose of providing software
    updates, detecting software piracy and verifying that customers are using
    Licensed Products in conformity with the applicable License Ke

/nfs/site/disks/psg_ctools_1/gcc/13.2.0/1/linux64/rhel7/bin/../lib/gcc/x86_64-none-linux/13.2.0/../../../../x86_64-none-linux/bin/ld: warning: rmar_nd.o: missing .note.GNU-stack section implies executable stack
/nfs/site/disks/psg_ctools_1/gcc/13.2.0/1/linux64/rhel7/bin/../lib/gcc/x86_64-none-linux/13.2.0/../../../../x86_64-none-linux/bin/ld: NOTE: This behaviour is deprecated and will be removed in a future version of the linker


rm -f _cuarc*.so _csrc*.so pre_vcsobj_*.so share_vcsobj_*.so
if [ -x ../simv ]; then chmod a-x ../simv; fi
g++  -o ../simv   -no-pie    -rdynamic  -Wl,-rpath='$ORIGIN'/simv.daidir -Wl,-rpath=./simv.daidir -Wl,-rpath=/p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib -L/p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib  -Wl,-rpath-link=./   objs/amcQw_d.o   _32616_archive_1.so   SIM_l.o       rmapats_mop.o rmapats.o rmar.o rmar_nd.o  rmar_llvm_0_1.o rmar_llvm_0_0.o            -lnuma -lvirsim -lerrorinf -lsnpsmalloc -lvfs    -lvcsnew -lsimprofile -lreader_common /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/libBA.a -luclinative /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/vcs_tls.o   -Wl,-whole-archive  -lvcsucli    -Wl,-no-whole-archive       ./../simv.daidir/vc_hdrs.o    /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/vcs_save_restore_new.o -ldl  -lc -lm -lpthread -ldl 
/nfs/site/disks/psg_ctools_1